In [1]:
# external
import numpy as np
import healpy as hp
import time
import ducc0
from tqdm import trange
# modules from cmblensplus
import cmblensplus.curvedsky as cs
import constant as c
import cmb

In [2]:
def benchmark(func, niter, desc):
    times = []

    for _ in trange(niter, desc=desc):
        t0 = time.perf_counter()
        func()
        times.append(time.perf_counter() - t0)

    times = np.asarray(times)

    print(f"{desc}")
    print(f"median: {np.median(times)}")
    print(f"mean:   {np.mean(times)}")
    print(f"min:    {np.min(times)}")


In [3]:
niter = 10

In [4]:
Lmax = 2*2048
# ucl is an array of shape [0:5,0:rlmax+1] and ucl[0,:] = TT, ucl[1,:] = EE, ucl[2,:] = TE, lcl[3,:] = phiphi, lcl[4,:] = Tphi
ucl = cmb.read_camb_cls('../data/unlensedcls.dat',ftype='scal',output='array')[:,:Lmax+1]

Generate random gaussian fields

In [6]:
Talm = cs.utils.gauss1alm(ucl[0,:])

In [7]:
alm = hp.synalm(ucl[0,:], lmax=Lmax, new=True)

Set parameters

In [8]:
nside = 1024  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside

In [9]:
Tmap = cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1])

In [10]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [11]:
benchmark(
    lambda: cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1]),
    niter=niter,
    desc="cs hp_alm2map",
)

cs hp_alm2map: 100%|██████████| 10/10 [00:09<00:00,  1.10it/s]

cs hp_alm2map
median: 0.9137365675414912
mean:   0.9114649165072478
min:    0.8893729289993644


In [12]:
alm_ducc = alm.reshape(1, -1)
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.synthesis(alm=alm_ducc,map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 synthesis",
)

ducc0 synthesis: 100%|██████████| 10/10 [00:07<00:00,  1.28it/s]

ducc0 synthesis
median: 0.7796902740374207
mean:   0.7803903019055725
min:    0.7531197170028463


In [13]:
benchmark(
    lambda: cs.utils.hp_map2alm(lmax,lmax,Tmap),
    niter=niter,
    desc="cs hp_map2alm",
)

cs hp_map2alm: 100%|██████████| 10/10 [00:09<00:00,  1.08it/s]

cs hp_map2alm
median: 0.918418807501439
mean:   0.9212634881027043
min:    0.8964522789465263


In [14]:
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.adjoint_synthesis(map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 adjoint_synthesis",
)

ducc0 adjoint_synthesis: 100%|██████████| 10/10 [00:08<00:00,  1.21it/s]

ducc0 adjoint_synthesis
median: 0.8191561379935592
mean:   0.8260194932227023
min:    0.8040159980300814


Higher resolution

In [15]:
nside = 2048  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside # Maximum multipole of harmonic coefficients to be computed

In [16]:
Tmap = cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1])

In [17]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [18]:
benchmark(
    lambda: cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1]),
    niter=niter,
    desc="cs hp_alm2map",
)

cs hp_alm2map: 100%|██████████| 10/10 [00:57<00:00,  5.78s/it]

cs hp_alm2map
median: 5.726992346521001
mean:   5.779801707807929
min:    5.6967748700408265


In [19]:
alm_ducc = alm.reshape(1, -1)
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.synthesis(alm=alm_ducc,map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 synthesis",
)

ducc0 synthesis: 100%|██████████| 10/10 [00:53<00:00,  5.30s/it]

ducc0 synthesis
median: 5.255041027558036
mean:   5.299405708990525
min:    5.178247214993462


In [20]:
benchmark(
    lambda: cs.utils.hp_map2alm(lmax,lmax,Tmap),
    niter=niter,
    desc="cs hp_map2alm",
)

cs hp_map2alm: 100%|██████████| 10/10 [01:02<00:00,  6.27s/it]

cs hp_map2alm
median: 6.270029510953464
mean:   6.270860145287588
min:    6.185553060029633


In [21]:
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.adjoint_synthesis(map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 adjoint_synthesis",
)

ducc0 adjoint_synthesis: 100%|██████████| 10/10 [00:54<00:00,  5.44s/it]

ducc0 adjoint_synthesis
median: 5.419578355969861
mean:   5.435537069803104
min:    5.384688913007267
